# 03 — Dataset de séquences

Projet GPT reproductible · Notebook en français.


## 1. Research objective

Question : comment former les exemples supervisés next-token ? Hypothèse : cible décalée d’un token et de même longueur que l’entrée.


## 2. Configuration

Seed 2026; device auto; contexte 8; stride 4; batch 2. Split des documents avant fenêtres pour limiter les fuites.


## 3. Implementation

GPTDataset retourne des tenseurs entiers (entrée,cible). Stride fixe le chevauchement.


In [1]:
import random,torch
from torch.utils.data import Dataset,DataLoader
SEED=2026; torch.manual_seed(SEED); random.seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class GPTDataset(Dataset):
 def __init__(self,token_ids,context_length,stride):
  if context_length<=0 or stride<=0: raise ValueError('Paramètres positifs requis')
  if len(token_ids)<=context_length: raise ValueError('Flux trop court')
  self.x=[]; self.y=[]
  for i in range(0,len(token_ids)-context_length,stride):
   a=token_ids[i:i+context_length]; b=token_ids[i+1:i+context_length+1]
   if len(a)==context_length and len(b)==context_length:
    self.x.append(torch.tensor(a,dtype=torch.long)); self.y.append(torch.tensor(b,dtype=torch.long))
 def __len__(self): return len(self.x)
 def __getitem__(self,i): return self.x[i],self.y[i]
tokens=list(range(40)); ds=GPTDataset(tokens,8,4)
loader=DataLoader(ds,batch_size=2,shuffle=True,generator=torch.Generator().manual_seed(SEED))
xb,yb=next(iter(loader)); print(DEVICE,xb.shape,yb.shape)


cuda torch.Size([2, 8]) torch.Size([2, 8])


## 4. Unit tests

Vérifier le décalage exact, les dimensions/types, les cas limites.


In [2]:
d=GPTDataset(list(range(12)),4,3)
assert len(d)==3 and d[0][0].tolist()==[0,1,2,3] and d[0][1].tolist()==[1,2,3,4]
assert all(x.dtype==y.dtype==torch.long for x,y in d)
for args in [(0,1),(4,0)]:
 try: GPTDataset(list(range(12)),*args)
 except ValueError: pass
 else: raise AssertionError('Paramètre invalide')
print('Tests dataset/décalage: OK')


Tests dataset/décalage: OK


## 5. Experiments

Comparer le nombre de fenêtres pour plusieurs strides.


In [3]:
[{'stride':s,'windows':len(GPTDataset(tokens,8,s))} for s in (1,4,8)]


[{'stride': 1, 'windows': 32},
 {'stride': 4, 'windows': 8},
 {'stride': 8, 'windows': 4}]

## 6. Results

Afficher forme du batch et première paire.


In [4]:
{"fenêtres": len(ds), "batch_entrée": tuple(xb.shape), "batch_cible": tuple(yb.shape), "première_paire": (ds[0][0].tolist(), ds[0][1].tolist())}


{'fenêtres': 8,
 'batch_entrée': (2, 8),
 'batch_cible': (2, 8),
 'première_paire': ([0, 1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7, 8])}

## 7. Discussion

Petit stride augmente les fenêtres corrélées. Pour grands corpus, utiliser un chargement indexé ou disque.
